In [1]:
import os
import shutil
import numpy as np
import nibabel as nib
from nilearn.image import resample_to_img

def resample_to_reference(
    ref_path: str,
    base_dir: str,
    target_names,
    *,
    overwrite: bool = False):

    """
    Resample selected NIfTI volumes to the voxel grid of a reference image
    **without turning missing data (NaNs) into valid zeros**.

    Parameters
    ----------
    ref_path : str
        Path to the reference NIfTI file whose affine and 3-D shape define
        the target grid.

    base_dir : str
        Root directory that will be searched (recursively) for files whose
        *file names* appear in `target_names`.

    target_names : Sequence[str]
        Exact file names (not full paths) that should be resampled whenever
        encountered under `base_dir`.

    overwrite : bool, default False
        If False (default) a resampled file that already exists is skipped.
        If True any existing file whose name ends with ``_resampled.nii`` or
        ``_resampled.nii.gz`` is deleted and replaced.

    Workflow
    --------
    1.  Build a boolean mask of *finite* voxels in the source volume
        (True ↔ valid data; False ↔ NaN/Inf).
    2.  Temporarily fill NaNs with 0 **only for interpolation**.
    3.  Resample the filled image (linear spline, “continuous”) and the mask
        (nearest-neighbour) to the reference grid.
    4.  Restore NaNs wherever the resampled mask is 0, guaranteeing that
        missing data remain missing and cannot bias later statistics.
    5.  Save the result alongside the source file, appending
        ``*_resampled.nii(.gz)`` to its name.
    """

    # Load reference once
    ref_img      = nib.load(ref_path)
    ref_affine   = ref_img.affine

    for root, _, files in os.walk(base_dir):
        for fname in files:
            if fname not in target_names:
                continue

            in_path     = os.path.join(root, fname)
            fname_lower = fname.lower()
            if fname_lower.endswith(".nii.gz"):
                base, ext = fname[:-7], ".nii.gz"
            elif fname_lower.endswith(".nii"):
                base, ext = fname[:-4], ".nii"
            else:
                # not a NIfTI we handle
                continue

            out_fname = f"{base}_resampled{ext}"
            out_path  = os.path.join(root, out_fname)

            # Skip or replace existing output
            if os.path.exists(out_path):
                if overwrite:
                    os.remove(out_path)
                else:
                    print(f"[SKIP] {out_path} already present")
                    continue

            # -----------------------------------------------------------------
            # 1) Load source, create validity mask, and make a filled copy
            # -----------------------------------------------------------------

            try:
                src_img = nib.load(in_path)
                data    = src_img.get_fdata()          # triggers decompression
            except Exception as err:
                print(f"[ERROR] {in_path} → {err}.  Skipping.")
                continue

            valid_mask = np.isfinite(data)
            filled_img = nib.Nifti1Image(
                np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0),
                src_img.affine,
                src_img.header,
            )
            mask_img   = nib.Nifti1Image(
                valid_mask.astype(np.uint8),
                src_img.affine,
            )

            # -----------------------------------------------------------------
            # 2) Short-circuit if already on the desired grid
            # -----------------------------------------------------------------

            if (src_img.shape[:3] == ref_img.shape[:3]) and np.allclose(src_img.affine, ref_affine):
                print(f"[INFO] {in_path} is already on the reference grid, skipping resampling.")
                continue

            else:

                print(f"[INFO] Resampling {in_path} → {out_fname}")
                # Choose interpolation: "nearest" for obvious mask files,
                # otherwise continuous (linear B-spline)
                interp = ("nearest" if fname.lower().endswith(("mask.nii", "mask.nii.gz")) else "continuous")

                resampled_img = resample_to_img(
                    filled_img,
                    ref_img,
                    interpolation=interp,
                    force_resample=True,
                    copy_header=True,
                )
                mask_rs_img = resample_to_img(
                    mask_img,
                    ref_img,
                    interpolation="nearest",
                    force_resample=True,
                    copy_header=True,
                )

            # -----------------------------------------------------------------
            # 3) Restore NaNs outside the (resampled) validity mask
            # -----------------------------------------------------------------

            rs_data = resampled_img.get_fdata()
            rs_mask = mask_rs_img.get_fdata() > 0.5
            rs_data[~rs_mask] = np.nan

            final_img = nib.Nifti1Image(
                rs_data.astype(data.dtype, copy=False),
                resampled_img.affine,
                resampled_img.header,
            )
            nib.save(final_img, out_path)

    print("Done")

# patterns = ['mask.nii.gz','beta_decisions.nii.gz','spmT_decisions.nii.gz','con_decisions.nii.gz']
# reference_img = '../../data/preprocessed/fmri/sub-18001/func/func.nii'
# search_folder = '../../data/modeled/lsa_decision'

patterns      = ['mask.nii.gz','beta_decisions.nii.gz'] # which files to re-sample?
reference_img = '../../../data/preprocessed/fmri/sub-18001/func/func.nii' # what is the reference image?
search_folder = '../../../analyses/lsa_onset/glms' # where to search for files to re-sample?
resample_to_reference(reference_img, search_folder, patterns, overwrite=False)

[SKIP] ../../../analyses/lsa_onset/glms/sub-14/mask_resampled.nii.gz already present
[SKIP] ../../../analyses/lsa_onset/glms/sub-14/beta_decisions_resampled.nii.gz already present
[INFO] ../../../analyses/lsa_onset/glms/sub-18011/mask.nii.gz is already on the reference grid, skipping resampling.
[INFO] ../../../analyses/lsa_onset/glms/sub-18011/beta_decisions.nii.gz is already on the reference grid, skipping resampling.
[INFO] ../../../analyses/lsa_onset/glms/sub-18018/mask.nii.gz is already on the reference grid, skipping resampling.
[INFO] ../../../analyses/lsa_onset/glms/sub-18018/beta_decisions.nii.gz is already on the reference grid, skipping resampling.
[SKIP] ../../../analyses/lsa_onset/glms/sub-15/mask_resampled.nii.gz already present
[SKIP] ../../../analyses/lsa_onset/glms/sub-15/beta_decisions_resampled.nii.gz already present
[SKIP] ../../../analyses/lsa_onset/glms/sub-12/mask_resampled.nii.gz already present
[SKIP] ../../../analyses/lsa_onset/glms/sub-12/beta_decisions_resam